# Ilaria RVC on Kaggle

The notebook requests **GPU T4 x2** and **Internet**. If Kaggle ignores these settings on import, select them in **Settings**, then click **Run All** in the editor. Open **Ilaria RVC** in the output of step 4.

Keep this notebook private while its services are running.

Each new Kaggle session downloads the latest Ilaria RVC code from GitHub. To update an open session, save your work, stop the app, then rerun steps 1 through 4.

**Automatic model export.** While the session runs, finished voice models and added indexes are copied into `ilaria-shared/models` in Kaggle Output and FileBrowser. Download them or save the notebook outputs before the session ends. Kaggle can still pause or end idle sessions. The automatic export includes inference files. To resume interrupted training, use **Stop and save** in Ilaria RVC and the checkpoint export below.


### 1. Check GPU

In [ ]:
import codecs
from pathlib import Path
import os
import shutil
import subprocess
import sys

rot_47 = lambda encoded_text: "".join(
    [
        (
            chr(
                (ord(c) - (ord("a") if c.islower() else ord("A")) - 47) % 26
                + (ord("a") if c.islower() else ord("A"))
            )
            if c.isalpha()
            else c
        )
        for c in encoded_text
    ]
)

WORK = Path("/kaggle/working")
REPO_ROOT = WORK / rot_47("dgvmdv-mqx")
TOOLS = WORK / ".ilaria-notebook"
PYTHON = TOOLS / "venv/bin/python"
SOURCE_URL = rot_47(codecs.decode("pbbxa://oqbpcj.kwu/BpmAbqvomzF/Qtizqi-ZDK.oqb", "rot_13"))
BRANCH = "main"
def run(*args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

if not WORK.is_dir() or sys.platform != "linux":
    raise RuntimeError("Run this notebook on Kaggle.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable an NVIDIA GPU and restart the session.")
run("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")

active = globals().get("ui_process")
if active is not None and active.poll() is None:
    raise RuntimeError("Stop Ilaria RVC before updating the code.")

TOOLS.mkdir(exist_ok=True)
if not REPO_ROOT.exists():
    print("Downloading Ilaria RVC. Enable Internet in Kaggle settings.", flush=True)
    run("git", "clone", "--depth", "1", "--branch", BRANCH, SOURCE_URL, REPO_ROOT)
if not (REPO_ROOT / ".git").is_dir():
    raise RuntimeError(f"Rename the existing folder {REPO_ROOT} and rerun this cell.")
run("git", "-C", REPO_ROOT, "fetch", "--depth", "1", SOURCE_URL, BRANCH)
run("git", "-C", REPO_ROOT, "checkout", "--detach", "--force", "FETCH_HEAD")
ROOT = REPO_ROOT / "app" if (REPO_ROOT / "app").is_dir() else REPO_ROOT
MODELS = REPO_ROOT / "models"
MODELS.mkdir(exist_ok=True)
release = __import__("json").loads((ROOT / "ilaria/ilaria_version.json").read_text(encoding="utf-8"))["version"]
if not (ROOT / ".env").exists():
    shutil.copyfile(ROOT / ".env.example", ROOT / ".env")
for name in ("datasets", "InferOutput"):
    (ROOT / name).mkdir(exist_ok=True)
print(f"Ilaria RVC {release} is ready in {ROOT}.")


### 2. Install dependencies

In [ ]:
import time
from html import escape
from IPython.display import HTML, display
from threading import Event, Thread

def install_environment(report):
    install_marker = TOOLS / "environment-v8-cu128.ok"
    runtime_env = {**os.environ, "GRADIO_ANALYTICS_ENABLED": "False",
                   "OMP_NUM_THREADS": "2", "MPLBACKEND": "Agg"}
    report(0, "Checking system tools")
    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
        run("apt-get", "-q", "update")
        run("apt-get", "-qy", "install", "ffmpeg")
    for binary in ("ffmpeg", "ffprobe"):
        executable = ROOT / (binary + ".exe")
        if not executable.exists():
            executable.symlink_to(shutil.which(binary))
    if not all(shutil.which(binary) for binary in ("gcc", "g++", "make")):
        run("apt-get", "-q", "update")
        run("apt-get", "-qy", "install", "build-essential")

    if not (PYTHON.is_file() and install_marker.is_file()):
        report(1, "Preparing Python")
        bootstrap = TOOLS / "bootstrap"
        if not (bootstrap / "uv").is_dir():
            run(sys.executable, "-m", "pip", "install", "--quiet", "--target", bootstrap, "uv==0.9.10")
        uv_env = {
            **os.environ,
            "PYTHONPATH": str(bootstrap),
            "UV_PYTHON_INSTALL_DIR": str(TOOLS / "python"),
            "UV_CACHE_DIR": str(TOOLS / "uv-cache"),
        }
        if not PYTHON.is_file():
            run(sys.executable, "-m", "uv", "python", "install", "3.10", env=uv_env)
            run(sys.executable, "-m", "uv", "venv", "--python", "3.10", "--seed", TOOLS / "venv", env=uv_env)

        install_log = TOOLS / "install.log"

        def uv_install(*args):
            with install_log.open("a", encoding="utf-8") as log:
                result = subprocess.run(
                    [sys.executable, "-m", "uv", "pip", "install", "--python", str(PYTHON), *args],
                    env=uv_env, stdout=log, stderr=subprocess.STDOUT,
                )
            if result.returncode:
                print(install_log.read_text(encoding="utf-8", errors="replace")[-12000:], flush=True)
                raise RuntimeError(f"uv installation failed. Full log: {install_log}")

        def pip(*args):
            with install_log.open("a", encoding="utf-8") as log:
                result = subprocess.run(
                    [str(PYTHON), "-m", "pip", "--disable-pip-version-check", *args],
                    stdout=log, stderr=subprocess.STDOUT,
                )
            if result.returncode:
                with install_log.open("rb") as log:
                    size = log.seek(0, os.SEEK_END)
                    log.seek(max(0, size - 12000))
                    print(log.read().decode("utf-8", errors="replace"), flush=True)
                raise RuntimeError(f"Dependency installation failed. Full log: {install_log}")

        pip("install", "pip==24.0", "setuptools==69.5.1", "wheel==0.43.0", "numpy==1.23.5")
        report(2, "Installing CUDA PyTorch")
        uv_install("--index", "https://download.pytorch.org/whl/cu128",
            "torch==2.10.0+cu128", "torchaudio==2.10.0+cu128")
        report(3, "Preparing native audio dependencies")
        # Build native modules separately so compiler errors reach Kaggle's output.
        build_env = {**os.environ, "CC": shutil.which("gcc"), "CXX": shutil.which("g++"),
                     "CPLUS_INCLUDE_PATH": os.environ.get("CPLUS_INCLUDE_PATH", ""),
                     "MAX_JOBS": "2"}
        if not build_env["CPLUS_INCLUDE_PATH"]:
            build_env.pop("CPLUS_INCLUDE_PATH")
        # HuBERT uses Fairseq's CPU extensions. Keep CUDA enabled for the app.
        # Fairseq enables CUDA extensions whenever CUDA_HOME exists, even if empty.
        build_env.pop("CUDA_HOME", None)
        build_env.update(CUDA_VISIBLE_DEVICES="", FORCE_CUDA="0")
        # The PyPI sdist omits C++ files included in the upstream release.
        fairseq_source = "https://codeload.github.com/facebookresearch/fairseq/tar.gz/refs/tags/v0.12.2"
        for package, source in (("fairseq==0.12.2", fairseq_source), ("pyworld==0.3.5", "pyworld==0.3.5")):
            report(3 if package.startswith("fairseq") else 4, f"Preparing {package}")
            name, version = package.split("==", 1)
            installed = subprocess.run(
                [str(PYTHON), "-c",
                 "from importlib.metadata import version; import sys; print(version(sys.argv[1]))",
                 name],
                capture_output=True, text=True,
            )
            if installed.returncode == 0 and installed.stdout.strip() == version:
                print(f"Reusing {package}.", flush=True)
                continue
            cython_version = "0.29.37" if name == "fairseq" else "3.0.12"
            pip("install", f"Cython=={cython_version}")
            print(f"Building {package}...", flush=True)
            build = subprocess.run(
                [str(PYTHON), "-m", "pip", "install", "--no-build-isolation",
                 "--no-deps", "--verbose", source],
                env=build_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            )
            with install_log.open("a", encoding="utf-8") as log:
                log.write(build.stdout)
            if build.returncode:
                print(build.stdout[-24000:], flush=True)
                raise RuntimeError(f"Native build failed for {package}. See the compiler output above.")
        report(5, "Installing audio and web interface dependencies")
        uv_install("--no-build-isolation",
            "gradio==3.14.0", "fastapi==0.88.0", "starlette==0.22.0", "pydantic==1.10.2",
            "uvicorn==0.20.0", "httpx==0.23.0", "python-multipart==0.0.5", "websockets==10.4",
            "altair==4.2.2", "jinja2==3.1.4", "pandas==1.5.3", "matplotlib==3.6.2", "pillow==9.5.0",
            "markdown-it-py==2.2.0", "mdit-py-plugins==0.3.3", "linkify-it-py==1.0.3",
            "numpy==1.23.5", "scipy==1.9.3", "numba==0.56.4", "llvmlite==0.39.1", "librosa==0.9.1",
            "scikit-learn==1.2.2", "resampy==0.4.3", "soundfile==0.12.1", "av==12.3.0",
            "faiss-cpu==1.7.4", "praat-parselmouth==0.4.6", "pyworld==0.3.5",
            "tensorboard==2.12.0", "tensorboardX==2.6.2.2", "protobuf==3.20.3", "onnx==1.15.0", "onnxruntime==1.16.3",
            "torchcrepe==0.0.20", "noisereduce==3.0.3", "python-dotenv==1.0.1",
            "tabulate==0.9.0", "pycryptodome==3.23.0", "psutil==6.1.1")
        # Fairseq uses legacy OmegaConf metadata handled by the pinned pip.
        pip("install", "--no-build-isolation",
            "fairseq==0.12.2", "hydra-core==1.0.7", "omegaconf==2.0.6")
        report(6, "Checking dependency versions")
        pip("check")
    else:
        print("Linux environment already installed.")

    report(7, "Testing GPU and audio imports")
    run(PYTHON, "-c", """
import torch
import torchaudio, fairseq, gradio, librosa, av, faiss, pyworld, parselmouth
from infer.modules.vc.modules import VC
from torch.utils.tensorboard import SummaryWriter
from tensorboardX import SummaryWriter as TensorBoardXWriter
assert torch.cuda.is_available(), "PyTorch cannot detect an NVIDIA GPU"
for gpu_id in range(torch.cuda.device_count()):
    print(f"GPU {gpu_id} ready: {torch.cuda.get_device_name(gpu_id)}")
print("PyTorch:", torch.__version__, "| Gradio:", gradio.__version__)
""", cwd=ROOT, env={**runtime_env, "PYTHONPATH": f"{ROOT / 'ilaria'}:{ROOT}"})
    install_marker.touch()
    return runtime_env

started = time.monotonic()
status = (0, "Preparing installation")
finished = Event()

def report(step, message):
    global status
    status = (step, message)

def panel():
    step, message = status
    elapsed = int(time.monotonic() - started)
    minutes, seconds = divmod(elapsed, 60)
    percent = step * 100 // 8
    return HTML(
        '<div style="max-width:640px;padding:16px 20px;border-radius:14px;'
        'background:#33243d;color:#fff1fa;border:1px solid #b68cbf">'
        f'<strong>Installing Ilaria RVC</strong> &nbsp; {percent}%'
        f'<progress aria-label="Installation progress" value="{percent}" max="100" '
        'style="display:block;width:100%;height:18px;accent-color:#ffc0cb;margin:8px 0"></progress>'
        f'<div>{escape(message)} &middot; Elapsed {minutes:02d}:{seconds:02d}</div></div>'
    )

view = display(panel(), display_id=True)

def refresh_display():
    while not finished.wait(1):
        view.update(panel())

timer = Thread(target=refresh_display, daemon=True)
timer.start()
try:
    runtime_env = install_environment(report)
except BaseException:
    report(status[0], "Installation stopped. See the output below.")
    raise
else:
    report(8, "Ready")
finally:
    finished.set()
    timer.join()
    view.update(panel())


### 3. Download model weights

In [ ]:
import hashlib
import time
from urllib.request import Request, urlopen

TRAINING_SAMPLE_RATES = ["32k"]  # [] for inference only, or ["32k", "40k", "48k"]
DOWNLOAD_FCPE = True
ASSET_SOURCE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/e6d0c1a17da07c33557852f9dfa2bd44cc75737d"
PRETRAIN_HASHES = {
    "32k": ("2332611297b8d88c7436de8f17ef5f07a2119353e962cd93cda5806d59a1133d",
            "bd7134e7793674c85474d5145d2d982e3c5d8124fc7bb6c20f710ed65808fa8a"),
    "40k": ("3b2c44035e782c4b14ddc0bede9e2f4a724d025cd073f736d4f43708453adfcb",
            "6b6ab091e70801b28e3f41f335f2fc5f3f35c75b39ae2628d419644ec2b0fa09"),
    "48k": ("b5d51f589cc3632d4eae36a315b4179397695042edc01d15312e1bddc2b764a4",
            "2269b73c7a4cf34da09aea99274dabf99b2ddb8a42cbfb065fb3c0aa9a2fc748"),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download(url, destination, expected_hash):
    if destination.is_file() and sha256_file(destination) == expected_hash:
        print(f"{destination.name}: already verified.")
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(1, 4):
        print(f"Downloading {destination.name} (attempt {attempt}/3)...", flush=True)
        try:
            with urlopen(Request(url, headers={"User-Agent": "Ilaria-RVC-Kaggle/2.1"}), timeout=90) as response:
                with temporary.open("wb") as stream:
                    shutil.copyfileobj(response, stream, length=4 * 1024 * 1024)
            if sha256_file(temporary) != expected_hash:
                raise ValueError(f"SHA-256 check failed for {destination.name}.")
            temporary.replace(destination)
            print(f"{destination.name}: ready.")
            return
        except (OSError, ValueError):
            if attempt == 3:
                raise
            time.sleep(2 * attempt)
        finally:
            temporary.unlink(missing_ok=True)

weights = [
    ("hubert_base.pt", "assets/hubert/hubert_base.pt",
     "f54b40fd2802423a5643779c4861af1e9ee9c1564dc9d32f54f20b5ffba7db96"),
    ("rmvpe.pt", "assets/rmvpe/rmvpe.pt",
     "6d62215f4306e3ca278246188607209f09af3dc77ed4232efdd069798c4ec193"),
]
for rate in TRAINING_SAMPLE_RATES:
    for role, digest in zip(("G", "D"), PRETRAIN_HASHES[rate]):
        filename = f"pretrained_v2/f0{role}{rate}.pth"
        weights.append((filename, f"assets/{filename}", digest))
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=3) as executor:
    downloads = [
        executor.submit(download, f"{ASSET_SOURCE}/{filename}", ROOT / relative_destination, digest)
        for filename, relative_destination, digest in dict.fromkeys(weights)
    ]
    for task in downloads:
        task.result()

if DOWNLOAD_FCPE:
    from zipfile import ZipFile
    fcpe_target = ROOT / "assets/pitch/fcpe_runtime/torchfcpe/assets/fcpe_c_v001.pt"
    fcpe_hash = "b9aeaeb673436eeda50ceafd632aa681aa63417e52eae4207503d180c9b10015"
    if not fcpe_target.is_file() or sha256_file(fcpe_target) != fcpe_hash:
        fcpe_wheel = TOOLS / "torchfcpe-0.0.4-py3-none-any.whl"
        fcpe_url = (
            "https://files.pythonhosted.org/packages/15/98/"
            "66da447498a2835b01cb3ae851e1333035a9fbf228e5c6f3e3c4351c06b6/"
            "torchfcpe-0.0.4-py3-none-any.whl"
        )
        try:
            download(fcpe_url, fcpe_wheel, "f042c463d850d76c6f4899a0b84f0b694bb560adf05f4de951097a756d17472d")
            fcpe_target.parent.mkdir(parents=True, exist_ok=True)
            temporary = fcpe_target.with_suffix(".part")
            try:
                with ZipFile(fcpe_wheel) as archive, archive.open("torchfcpe/assets/fcpe_c_v001.pt") as model:
                    with temporary.open("wb") as stream:
                        shutil.copyfileobj(model, stream)
                if sha256_file(temporary) != fcpe_hash:
                    raise ValueError("SHA-256 check failed for FCPE.")
                temporary.replace(fcpe_target)
            finally:
                temporary.unlink(missing_ok=True)
        except (OSError, KeyError, ValueError) as error:
            print(f"FCPE unavailable ({error}). Use RMVPE.")
print("Base weights ready.")

## Start

### Access Setup

Keep the notebook private while these services are running, and do not publish saved cell output containing the generated credentials.

- **Ngrok** uses Ngrok tunnels for the WebUI and FileBrowser. Enter your auth token from https://dashboard.ngrok.com/tunnels/authtokens.
- **Gradio + Cloudflare** uses Gradio for the WebUI and Cloudflare Quick Tunnel for FileBrowser.
- **Horizon** provides separate WebUI and FileBrowser links. Enter your Horizon ID from https://hrzn.run/dashboard/ and approve the CLI token request when prompted.

Both services have separate session passwords. FileBrowser exposes only the dedicated export folder. Run the stop cell to close both services and their tunnels.

### Training performance

- **Training precision** uses FP16 in Auto mode on T4 and native BF16 on compatible GeForce RTX 30/40/50 GPUs. Use FP32 for a baseline, or match the precision of an existing run before resuming it. This controls the initial **Training Precision** selection in the WebUI, where you can still change it.
- **Workers and prefetch** default to 2 each. Set them to 4 and 8 respectively to compare against the previous loader configuration. **Fused AdamW** is off by default. Turn it on to benchmark it on a new run.
- For fresh runs on this version, **Batch Size** is per GPU. Compare one T4 at 8 with two T4s at 4 for a global batch of 8. Use the same prepared dataset, pretrain and precision, and compare after the first few epochs. Keep the original batch configuration when resuming a checkpoint.


Training Log shows the effective batch sizes and steps per epoch. For comparison runs, check Training timing in train.log for batch time, loader wait, and checkpoint/preview time.


In [ ]:
import re
import secrets
import socket
import tarfile
from tempfile import TemporaryDirectory

TUNNEL = "Gradio + Cloudflare"  # or "Ngrok", "Horizon"
NGROK_AUTHTOKEN = ""
HORIZON_ID = ""
WEBUI_PASSWORD = "ilaria"
FILEBROWSER_USER = "ilaria"
FILEBROWSER_PASSWORD = "ilaria123456"

# Training defaults for this Kaggle session.
TRAINING_PRECISION = "Auto"  # Auto: FP16 on T4, otherwise FP32. Or choose "FP16" / "FP32".
TRAINING_WORKERS = 2         # Workers per GPU. Use 4 for the previous baseline.
TRAINING_PREFETCH = 2        # Batches per worker. Use 8 for the previous baseline.
FUSED_ADAMW = False          # Set True to benchmark fused AdamW on CUDA.
import signal
from html import escape
from IPython.display import HTML, Javascript, display

if TRAINING_PRECISION not in {"Auto", "BF16", "FP16", "FP32"}:
    raise ValueError("Training precision must be Auto, BF16, FP16, or FP32.")
if TRAINING_WORKERS not in {2, 4} or TRAINING_PREFETCH not in {2, 8}:
    raise ValueError("Choose 2 or 4 workers and a prefetch of 2 or 8.")
if not isinstance(FUSED_ADAMW, bool):
    raise ValueError("FUSED_ADAMW must be True or False.")

runtime_env = {
    **runtime_env,
    "ILARIA_TRAIN_PRECISION": TRAINING_PRECISION,
    "ILARIA_TRAIN_WORKERS": str(TRAINING_WORKERS),
    "ILARIA_TRAIN_PREFETCH": str(TRAINING_PREFETCH),
    "ILARIA_TRAIN_FUSED_ADAMW": "1" if FUSED_ADAMW else "0",
}

# PyAV accepts r/w even when the underlying stream is binary.
audio_module = ROOT / "infer/lib/audio.py"
audio_source = audio_module.read_text(encoding="utf-8")
patched_audio = audio_source.replace('av.open(i, "rb")', 'av.open(i, "r")').replace(
    'av.open(o, "wb",', 'av.open(o, "w",'
)
if patched_audio != audio_source:
    audio_module.write_text(patched_audio, encoding="utf-8")

# Decode path-based inference and training audio with Kaggle's FFmpeg.
# PyAV can inherit "rb" from the Python file object's mode.
decoder = """
def load_audio(file, sr):
    import subprocess

    path = file.strip().strip('"').strip()
    result = subprocess.run(
        ["ffmpeg", "-nostdin", "-v", "error", "-i", path,
         "-map", "0:a:0", "-ac", "1", "-ar", str(sr),
         "-c:a", "pcm_f32le", "-f", "f32le", "pipe:1"],
        capture_output=True,
    )
    if result.returncode:
        detail = result.stderr.decode("utf-8", errors="replace").strip()
        raise RuntimeError(f"Could not decode audio: {detail}")
    return np.frombuffer(result.stdout, dtype="<f4").copy()
"""
audio_source = audio_module.read_text(encoding="utf-8")
if decoder.strip() not in audio_source:
    start = audio_source.index("def load_audio(file, sr):")
    end = audio_source.find("\ndef ", start + 1)
    if end < 0:
        end = len(audio_source)
    patched_audio = audio_source[:start] + decoder.strip() + "\n\n" + audio_source[end:]
    compile(patched_audio, str(audio_module), "exec")
    audio_module.write_text(patched_audio, encoding="utf-8")

# Apply training improvements to this Kaggle checkout only.
training_patch = r'''
diff --git a/ILARIA_RVC.py b/ILARIA_RVC.py
index adf7b58..9139f26 100644
--- a/ILARIA_RVC.py
+++ b/ILARIA_RVC.py
@@ -991,23 +991,25 @@ def stream_training_progress(process, log_path, total_epochs, offset=0, total_st
     completed_steps = 0
     pending = ""
     epoch_line = ""
+    batch_line = ""
     remaining = "measuring first epoch"
     estimated_finish = "calculating..."
     completion_marker = None
     stopped_marker = None
 
     def status(state="Training Info", completed_at=None, inactive=False):
+        run_details = "\n" + batch_line if batch_line and not inactive and not completed_at else ""
         if target_steps is not None:
             return ("%s: %.1f%% | Epochs: %s completed | Current Epoch: %s | Total Steps: %s/%s"
                     % (state, min(100.0, completed_steps / target_steps * 100.0), completed, current_epoch, completed_steps, target_steps)
                     + (" | Completed at: " + completed_at if completed_at else "" if inactive else
                        " | Remaining: %s | Estimated finish: %s" % (remaining, estimated_finish))
-                    + ("\n" + epoch_line if epoch_line else ""))
+                    + ("\n" + epoch_line if epoch_line else "") + run_details)
         return ("%s: %.1f%% | Epochs: %s/%s | Total Steps: %s"
                 % (state, completed / total_epochs * 100, completed, total_epochs, completed_steps)
                 + (" | Completed at: " + completed_at if completed_at else "" if inactive else
                    " | Remaining: %s | Estimated finish: %s" % (remaining, estimated_finish))
-                + ("\n" + epoch_line if epoch_line else ""))
+                + ("\n" + epoch_line if epoch_line else "") + run_details)
 
     preview_manifest = pathlib.Path(log_path).parent / "previews" / "previews.json"
     preview_modified = None
@@ -1032,6 +1034,10 @@ def stream_training_progress(process, log_path, total_epochs, offset=0, total_st
             lines.append(pending)
             pending = ""
         for line in lines:
+            batch = re.search(r"Training batch: (\d+) per GPU \| (\d+) total \| (\d+) steps/epoch", line)
+            if batch:
+                batch_line = "Batch: %s per GPU | %s total | %s steps/epoch" % batch.groups()
+                yield status()
             resumed = re.search(r"RESUMED! Epochs: (\d+) \| Total Steps: (\d+)", line)
             if resumed:
                 completed = int(resumed.group(1)) if target_steps is not None else min(total_epochs, int(resumed.group(1)))
diff --git a/ilaria/ilaria_training_state.py b/ilaria/ilaria_training_state.py
index a22b5e9..e7ea6da 100644
--- a/ilaria/ilaria_training_state.py
+++ b/ilaria/ilaria_training_state.py
@@ -20,6 +20,8 @@ import torch.distributed as dist
 
 STATE_DIR = "training-state"
 STOP_FILE = "stop-request.json"
+PER_GPU_BATCH = "per_gpu"
+LEGACY_BATCH = "legacy"
 _GENERATION = re.compile(r"e\d+-s\d+-[a-f0-9]{32}\Z")
 
 
@@ -31,7 +33,21 @@ def _plain(value):
     return value
 
 
-def training_signature(hps, world_size):
+def training_batch_layout(model_dir):
+    """Retain the batch grouping of checkpoints written before the sampler fix."""
+    root = Path(model_dir)
+    manifest = root / STATE_DIR / "latest.json"
+    if manifest.is_file():
+        layout = json.loads(manifest.read_text(encoding="utf-8")).get("batch_layout", LEGACY_BATCH)
+        if layout not in (PER_GPU_BATCH, LEGACY_BATCH):
+            raise ValueError("Unknown batch layout in saved training state.")
+        return layout
+    if any(root.glob("G_*.pth")) or any(root.glob("D_*.pth")):
+        return LEGACY_BATCH
+    return PER_GPU_BATCH
+
+
+def training_signature(hps, world_size, batch_layout=LEGACY_BATCH):
     """Allow a larger target epoch count, but never resume a different experiment."""
     filelist = Path(hps.data.training_files).read_bytes()
     signature = {
@@ -42,6 +58,8 @@ def training_signature(hps, world_size):
         "batch_size": int(hps.train.batch_size), "world_size": int(world_size),
         "filelist_sha256": hashlib.sha256(filelist).hexdigest(),
     }
+    if world_size > 1 and batch_layout != LEGACY_BATCH:
+        signature["batch_layout"] = batch_layout
     # Preserve compatibility with old default-architecture checkpoints.
     for key, default in (("vocoder", "HiFi-GAN"), ("embedder_model", "contentvec")):
         value = getattr(hps.train, key, default)
@@ -162,6 +180,8 @@ def save_training_state(model_dir, nets, optims, schedulers, scaler,
                     "batch_offset": int(batch_offset),
                     "files": {name: _sha256(destination / name)
                               for name in ("G.pth", "D.pth", "training.pt")}}
+        if "batch_layout" in signature:
+            manifest["batch_layout"] = signature["batch_layout"]
         with temporary.open("x", encoding="utf-8") as stream:
             json.dump(manifest, stream, indent=2)
             stream.write("\n")
diff --git a/infer/lib/infer_pack/commons.py b/infer/lib/infer_pack/commons.py
index 05ea342..458f96f 100644
--- a/infer/lib/infer_pack/commons.py
+++ b/infer/lib/infer_pack/commons.py
@@ -43,22 +43,32 @@ def rand_gumbel_like(x):
     return g
 
 
-def slice_segments(x, ids_str, segment_size=4):
-    ret = torch.zeros_like(x[:, :, :segment_size])
-    for i in range(x.size(0)):
-        idx_str = ids_str[i]
-        idx_end = idx_str + segment_size
-        ret[i] = x[i, :, idx_str:idx_end]
-    return ret
-
-
-def slice_segments2(x, ids_str, segment_size=4):
-    ret = torch.zeros_like(x[:, :segment_size])
-    for i in range(x.size(0)):
-        idx_str = ids_str[i]
-        idx_end = idx_str + segment_size
-        ret[i] = x[i, idx_str:idx_end]
-    return ret
+def _segment_indices(
+    x: torch.Tensor, starts: torch.Tensor, segment_size: int
+) -> torch.Tensor:
+    """Build batched indices without reading individual CUDA scalars on the CPU."""
+    length = x.size(-1)
+    width = x[..., :segment_size].size(-1)
+    starts = starts[: x.size(0)].reshape(x.size(0)).to(device=x.device, dtype=torch.long)
+    ends = starts + segment_size
+    # Match Python's handling of negative slice bounds and the former assignment's
+    # broadcast when a slice contains just one value.
+    starts = torch.where(starts < 0, starts + length, starts).clamp(0, length)
+    ends = torch.where(ends < 0, ends + length, ends).clamp(0, length)
+    sizes = (ends - starts).clamp_min(0)
+    # Incomplete slices used to fail assignment. Leave their indices out of bounds.
+    starts = torch.where((sizes == width) | (sizes == 1), starts, length)
+    offsets = torch.arange(width, device=x.device)
+    return starts.unsqueeze(1) + (sizes != 1).unsqueeze(1) * offsets
+
+
+def slice_segments(x: torch.Tensor, ids_str: torch.Tensor, segment_size: int = 4):
+    indices = _segment_indices(x, ids_str, segment_size).unsqueeze(1)
+    return torch.gather(x, 2, indices.expand(-1, x.size(1), -1))
+
+
+def slice_segments2(x: torch.Tensor, ids_str: torch.Tensor, segment_size: int = 4):
+    return torch.gather(x, 1, _segment_indices(x, ids_str, segment_size))
 
 
 def rand_slice_segments(x, x_lengths=None, segment_size=4):
diff --git a/infer/modules/train/train.py b/infer/modules/train/train.py
index 2ed59c1..5d57eea 100644
--- a/infer/modules/train/train.py
+++ b/infer/modules/train/train.py
@@ -7,6 +7,7 @@ _ilaria_sys.path.append(str(_ilaria_install_root))
 import os
 import sys
 import logging
+from time import perf_counter
 from contextlib import contextmanager
 
 logger = logging.getLogger(__name__)
@@ -23,6 +24,7 @@ from ilaria_training_state import (
     training_signature, has_training_state, load_training_state,
     restore_runtime_state, epoch_stop_requested, gather_rng_states,
     save_training_state, share_boundary_error, distributed,
+    LEGACY_BATCH, training_batch_layout,
 )
 from ilaria_training_preview import TrainingPreview
 from ilaria_training_targets import step_target, target_epoch_limit, StepETA
@@ -112,16 +114,19 @@ class EpochRecorder:
 
 @contextmanager
 def frozen_discriminator(model):
-    """Keep gradients through fake audio, without unused single-device D gradients."""
-    # Keep the established reducer behavior for multi-GPU training.
+    """Preserve the gradient to fake audio during the generator update."""
     if isinstance(model, DDP):
-        yield
-        return
+        # Bypass the reducer for this forward. Keep DDP's buffer broadcast
+        # for discriminators with spectral normalization.
+        if model.broadcast_buffers:
+            for buffer in model.module.buffers():
+                dist.broadcast(buffer, src=0, group=model.process_group)
+        model = model.module
     parameters = tuple((parameter, parameter.requires_grad) for parameter in model.parameters())
     try:
         for parameter, _ in parameters:
             parameter.requires_grad_(False)
-        yield
+        yield model
     finally:
         for parameter, requires_grad in parameters:
             parameter.requires_grad_(requires_grad)
@@ -181,45 +186,69 @@ def run(
         writer.flush()
         writer_eval = SummaryWriter(log_dir=os.path.join(hps.model_dir, "eval"))
 
+    cuda_available = torch.cuda.is_available()
+    if cuda_available:
+        torch.cuda.set_device(rank)
+    # NCCL uses the rank's current CUDA device, including for object collectives.
     # A single device needs neither a process group nor DDP reducers.
     if n_gpus > 1:
+        use_nccl = (
+            sys.platform.startswith("linux")
+            and cuda_available
+            and torch.version.cuda is not None
+            and dist.is_nccl_available()
+        )
+        backend = "nccl" if use_nccl else "gloo"
         dist.init_process_group(
-            backend="gloo", init_method="env://", world_size=n_gpus, rank=rank
+            backend=backend, init_method="env://", world_size=n_gpus, rank=rank
         )
+        if rank == 0:
+            logger.info("Training distributed backend: %s", backend)
     torch.manual_seed(hps.train.seed)
-    if torch.cuda.is_available():
-        torch.cuda.set_device(rank)
 
     if hps.if_f0 == 1:
         train_dataset = TextAudioLoaderMultiNSFsid(hps.data.training_files, hps.data)
     else:
         train_dataset = TextAudioLoader(hps.data.training_files, hps.data)
+    batch_layout = training_batch_layout(hps.model_dir)
+    per_gpu_batch = hps.train.batch_size * n_gpus if batch_layout == LEGACY_BATCH else hps.train.batch_size
     train_sampler = DistributedBucketSampler(
         train_dataset,
-        hps.train.batch_size * n_gpus,
+        per_gpu_batch,
         # [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1200,1400],  # 16s
         [100, 200, 300, 400, 500, 600, 700, 800, 900],  # 16s
         num_replicas=n_gpus,
         rank=rank,
         shuffle=True,
     )
-    # It is possible that dataloader's workers are out of shared memory. Please try to raise your shared memory limit.
-    # num_workers=8 -> num_workers=4
+    loader_workers = int(os.environ.get("ILARIA_TRAIN_WORKERS", "4"))
+    prefetch_batches = int(os.environ.get("ILARIA_TRAIN_PREFETCH", "8"))
+    if loader_workers < 1 or prefetch_batches < 1:
+        raise ValueError("Training workers and prefetch must be positive.")
     if hps.if_f0 == 1:
         collate_fn = TextAudioCollateMultiNSFsid()
     else:
         collate_fn = TextAudioCollate()
     train_loader = DataLoader(
         train_dataset,
-        num_workers=4,
+        num_workers=loader_workers,
         shuffle=False,
         pin_memory=True,
         collate_fn=collate_fn,
         batch_sampler=train_sampler,
         persistent_workers=True,
         worker_init_fn=configure_training_process,
-        prefetch_factor=8,
+        prefetch_factor=prefetch_batches,
     )
+    if rank == 0:
+        unique_clips = sum(len(bucket) for bucket in train_sampler.buckets)
+        logger.info(
+            "Training batch: %d per GPU | %d total | %d steps/epoch | %d unique clips | %d repeated | %s",
+            per_gpu_batch, per_gpu_batch * n_gpus, len(train_loader),
+            unique_clips, train_sampler.total_size - unique_clips, batch_layout,
+        )
+        logger.info("Training loader: %d workers per rank | prefetch %d batches per worker",
+                    loader_workers, prefetch_batches)
     if hps.if_f0 == 1:
         net_g = RVC_Model_f0(
             hps.data.filter_length // 2 + 1,
@@ -243,17 +272,20 @@ def run(
              if selected_vocoder == "RefineGAN" else MultiPeriodDiscriminator(hps.model.use_spectral_norm))
     if torch.cuda.is_available():
         net_d = net_d.cuda(rank)
+    fused_adamw = torch.cuda.is_available() and os.environ.get("ILARIA_TRAIN_FUSED_ADAMW") == "1"
     optim_g = torch.optim.AdamW(
         net_g.parameters(),
         hps.train.learning_rate,
         betas=hps.train.betas,
         eps=hps.train.eps,
+        fused=fused_adamw,
     )
     optim_d = torch.optim.AdamW(
         net_d.parameters(),
         hps.train.learning_rate,
         betas=hps.train.betas,
         eps=hps.train.eps,
+        fused=fused_adamw,
     )
     if n_gpus > 1:
         if hasattr(torch, "xpu") and torch.xpu.is_available():
@@ -265,7 +297,7 @@ def run(
             net_g = DDP(net_g)
             net_d = DDP(net_d)
 
-    signature = training_signature(hps, n_gpus)
+    signature = training_signature(hps, n_gpus, batch_layout)
     saved_state = None
     target_steps = step_target(hps)
     if has_training_state(hps.model_dir):
@@ -313,6 +345,20 @@ def run(
                 pretrain = torch.load(hps.pretrainD, map_location="cpu", weights_only=True)
                 target.load_state_dict(weights_for_runtime(pretrain["model"]))
 
+    # Older torch versions restore the saved flags and leave AdamW step tensors on CPU.
+    for optimizer in (optim_g, optim_d):
+        for group in optimizer.param_groups:
+            group["fused"] = fused_adamw
+            if fused_adamw:
+                group["foreach"] = False
+            for parameter in group["params"]:
+                step = optimizer.state.get(parameter, {}).get("step")
+                if torch.is_tensor(step):
+                    device = parameter.device if fused_adamw or group.get("capturable") else "cpu"
+                    optimizer.state[parameter]["step"] = step.to(device)
+    if rank == 0:
+        logger.info("Training optimizer: AdamW%s", " fused" if fused_adamw else "")
+
     scheduler_g = torch.optim.lr_scheduler.ExponentialLR(
         optim_g, gamma=hps.train.lr_decay, last_epoch=-1 if saved_state is not None else epoch_str - 2
     )
@@ -486,9 +532,14 @@ def train_and_evaluate(
         # Loader
         data_iterator = enumerate(train_loader)
 
-    # Run steps
+    # Time the loader separately from the batch loop's compute and logging.
+    loader_wait = 0.0
+    batch_started = perf_counter()
+    wait_started = batch_started
     for batch_position, (batch_idx, info) in enumerate(data_iterator):
+        loader_wait += perf_counter() - wait_started
         if batch_position < resume_offset:
+            wait_started = perf_counter()
             continue
         log_step = rank == 0 and global_step % hps.train.log_interval == 0
         optim_g.zero_grad(set_to_none=True)
@@ -580,13 +631,12 @@ def train_and_evaluate(
             scaler.unscale_(optim_d)
             grad_norm_d = commons.clip_grad_value_(net_d.parameters(), None)
         scaler.step(optim_d)
-        if not isinstance(net_d, DDP):
-            optim_d.zero_grad(set_to_none=True)
+        optim_d.zero_grad(set_to_none=True)
 
-        with frozen_discriminator(net_d):
+        with frozen_discriminator(net_d) as discriminator_for_generator:
             with autocast():
                 # Generator
-                y_d_hat_r, y_d_hat_g, fmap_r, fmap_g = net_d(wave, y_hat)
+                _, y_d_hat_g, fmap_r, fmap_g = discriminator_for_generator(wave, y_hat)
                 with autocast(enabled=False):
                     loss_mel = (refinegan_mel_loss(wave, y_hat, hps.data.sampling_rate) * hps.train.c_mel / 3.0
                                 if getattr(hps.train, "vocoder", "HiFi-GAN") == "RefineGAN"
@@ -671,8 +721,14 @@ def train_and_evaluate(
                     logger.info(message)
             if global_step >= target_steps:
                 break
+        wait_started = perf_counter()
     # /Run steps
 
+    if rank == 0 and torch.cuda.is_available():
+        torch.cuda.synchronize()
+    batch_seconds = perf_counter() - batch_started
+    boundary_started = perf_counter()
+
     # Complete both learning-rate schedules before storing the boundary. The
     # resumed optimizer and scheduler therefore describe the NEXT epoch.
     partial_epoch = completed_batches < len(train_loader)
@@ -740,10 +796,12 @@ def train_and_evaluate(
     stopping = stopping or after_preview_stop
     if rank == 0:
         if epoch_eta is not None:
-            if torch.cuda.is_available():
-                torch.cuda.synchronize()
             epoch_eta.finish_epoch(min(epoch, hps.total_epoch))
             logger.info("Training Model: {} ====> Epoch: {} || Total Steps: {} || Time taken: {} | {}".format(hps.name, epoch, global_step, epoch_recorder.record(), epoch_eta.summary(include_epoch_time=False)))
+            logger.info(
+                "Training timing: %.2fs batches (%.2fs waiting for data) | %.2fs checkpoint/preview boundary",
+                batch_seconds, loader_wait, perf_counter() - boundary_started,
+            )
         else:
             logger.info("Training Model: {} ====> Epoch: {} || Total Steps: {} || Time taken: {}".format(hps.name, epoch, global_step, epoch_recorder.record()))
     # All ranks wait for the save/export result. No peer can start another
'''
training_patch = training_patch.lstrip("\n").encode("utf-8")
patch_command = ["git", "apply", "--ignore-whitespace", "--whitespace=nowarn"]
if "REPO_ROOT" in globals() and ROOT != REPO_ROOT:
    patch_command.append(f"--directory={ROOT.name}")
git_cwd = REPO_ROOT if "REPO_ROOT" in globals() else ROOT
already_applied = subprocess.run(
    [*patch_command, "--reverse", "--check"], cwd=git_cwd, input=training_patch,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
if already_applied.returncode != 0:
    result = subprocess.run(
        patch_command, cwd=git_cwd, input=training_patch, capture_output=True,
    )
    if result.returncode != 0:
        details = result.stderr.decode("utf-8", errors="replace").strip()
        raise RuntimeError(f"The Kaggle training patch does not match this checkout. Rerun step 1 to refresh it.\n{details}")

entrypoint = ROOT / "ILARIA_RVC.py"
original = "share=True, favicon_path=str(UI_ICON_PATH), prevent_thread_lock=True"
patched = (
    'share=True, favicon_path=str(UI_ICON_PATH), prevent_thread_lock=True,\n'
    '            server_name="127.0.0.1",\n'
)
source = entrypoint.read_text(encoding="utf-8")
previous_patch = patched.replace('server_name="127.0.0.1",', 'server_name="127.0.0.1", server_port=config.listen_port,')
if previous_patch in source:
    source = source.replace(previous_patch, patched, 1)
    entrypoint.write_text(source, encoding="utf-8")
if patched not in source and "ILARIA_WEBUI_PASSWORD" not in source:
    if source.count(original) != 1:
        raise RuntimeError("The launch entry point has changed.")
    entrypoint.write_text(source.replace(original, patched), encoding="utf-8")

secure_launch = patched.replace(
    "share=True,",
    'share=os.environ["ILARIA_KAGGLE_SHARE"] == "1",',
).replace(
    'server_name="127.0.0.1",',
    'server_name="127.0.0.1", auth=("ilaria", os.environ["ILARIA_WEBUI_PASSWORD"]),',
)
source = entrypoint.read_text(encoding="utf-8")
if secure_launch not in source:
    if source.count(patched) != 1:
        raise RuntimeError("The WebUI launch configuration has changed.")
    entrypoint.write_text(source.replace(patched, secure_launch, 1), encoding="utf-8")

# Limit work on the Kaggle runtime and reduce background tunnel traffic.
source = entrypoint.read_text(encoding="utf-8")
source = source.replace(
    "app.load(refresh_training_queue, [queue_job], [queue_table, queue_status, queue_job], every=2, show_progress=False)",
    "app.load(refresh_training_queue, [queue_job], [queue_table, queue_status, queue_job], every=10, show_progress=False)",
    1,
)
cloud_launch = "if config.iscolab:\n        app.queue(concurrency_count=511, max_size=1022).launch("
source = source.replace(
    cloud_launch,
    "if config.iscolab:\n        app.queue(concurrency_count=2, max_size=32).launch(",
    1,
)
# Use the existing streaming model importer through the Gradio tunnel.
source = source.replace(
    '                if not config.iscolab:\n                    build_binary_model_upload(gr)',
    '                build_binary_model_upload(gr)',
)
start = source.index('                # Keep the existing API available to integrations using Gradio File inputs.')
end = source.index('            with gr.Accordion(', start)
source = source[:start] + source[start:end].replace('visible=config.iscolab)', 'visible=False)') + source[end:]
source = source.replace(
    '    if not config.iscolab:\n        app.load(_js=BINARY_MODEL_UPLOAD_JS)',
    '    app.load(_js=BINARY_MODEL_UPLOAD_JS)',
)
source = source.replace(
    '    if not config.iscolab:\n        register_binary_model_import(app.server_app, now_dir, update_operation_gate)',
    '    register_binary_model_import(app.server_app, now_dir, update_operation_gate)\n    if not config.iscolab:',
)
upload_module = ROOT / "ilaria/ilaria_model_upload.py"
upload_source = upload_module.read_text(encoding="utf-8")
old_origin = '        if not origin or origin.casefold() != f"{request.url.scheme}://{request.url.netloc}".casefold():'
new_origin = """        allowed_origins = {f"{request.url.scheme}://{request.url.netloc}".casefold()}
        share_url = getattr(request.app.blocks, "share_url", None)
        if share_url:
            allowed_origins.add(share_url.rstrip("/").casefold())
        if not origin or origin.casefold() not in allowed_origins:"""
if new_origin not in upload_source:
    if upload_source.count(old_origin) != 1:
        raise RuntimeError("The model upload origin check has changed.")
    upload_source = upload_source.replace(old_origin, new_origin, 1)
    compile(upload_source, str(upload_module), "exec")
    upload_module.write_text(upload_source, encoding="utf-8")

# Set the CUDA pitch extractor for notebook training only.
pitch_start = source.index("f0method8 = gr.Radio(")
pitch_end = source.index("save_epoch10 = gr.Slider(", pitch_start)
pitch_controls = source[pitch_start:pitch_end]
pitch_controls = pitch_controls.replace('value="rmvpe",', 'value="rmvpe_gpu" if F0GPUVisible and gpus else "rmvpe",', 1)
pitch_controls = pitch_controls.replace('value="%s-%s" % (gpus, gpus),', 'value=gpus,', 1)
pitch_controls = pitch_controls.replace("visible=False,", "visible=bool(F0GPUVisible and gpus),", 1)
source = source[:pitch_start] + pitch_controls + source[pitch_end:]

source = source.replace("default_batch_size = 8", "default_batch_size = 4", 1)
compile(source, str(entrypoint), "exec")
entrypoint.write_text(source, encoding="utf-8")

# Keep the dashboard restricted to the local UI and its own Gradio link.
dashboard = ROOT / "ilaria/ilaria_training_dashboard.py"
local_origin_test = '        if request.headers.get("origin", "").casefold() != f"{request.url.scheme}://{request.url.netloc}".casefold():'
share_origin_test = (
    '        allowed_origins = {f"{request.url.scheme}://{request.url.netloc}".casefold()}\n'
    '        share_url = getattr(request.app.blocks, "share_url", None)\n'
    '        if share_url:\n'
    '            allowed_origins.add(share_url.rstrip("/").casefold())\n'
    '        if request.headers.get("origin", "").casefold() not in allowed_origins:'
)
source = dashboard.read_text(encoding="utf-8")
if source.count(share_origin_test) != 2:
    if source.count(local_origin_test) != 2:
        raise RuntimeError("The Training Dashboard access check has changed.")
    dashboard.write_text(source.replace(local_origin_test, share_origin_test), encoding="utf-8")

log_file = TOOLS / "startup.log"

def log_tail(lines=30):
    if not log_file.is_file():
        return ""
    with log_file.open("rb") as stream:
        size = stream.seek(0, os.SEEK_END)
        stream.seek(max(0, size - 65536))
        return "\n".join(stream.read().decode("utf-8", errors="replace").splitlines()[-lines:])

def stop_ui(process):
    if process is not None and process.poll() is None:
        for stop_signal, timeout in ((signal.SIGINT, 15), (signal.SIGTERM, 10), (signal.SIGKILL, 5)):
            try:
                os.killpg(process.pid, stop_signal)
            except ProcessLookupError:
                return
            try:
                process.wait(timeout=timeout)
                return
            except subprocess.TimeoutExpired:
                continue
        raise RuntimeError("The process is still running. Use Stop Session in Kaggle.")

def stop_access():
    if "ngrok" in globals():
        try:
            ngrok.kill()
        except Exception:
            pass
    for process in reversed(globals().get("access_processes", [])):
        stop_ui(process)

stop_access()
access_processes = []

def background(command, name):
    path = TOOLS / (name + ".log")
    with path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    access_processes.append(process)
    return process, path

def wait_for_url(process, path, pattern, timeout=90):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            break
        match = re.search(pattern, path.read_text(encoding="utf-8", errors="replace"))
        if match:
            return match[0]
        time.sleep(1)
    raise RuntimeError(f"Could not open {path.stem}. See {path}.")

def free_port():
    with socket.socket() as listener:
        listener.bind(("127.0.0.1", 0))
        return listener.getsockname()[1]

def start_filebrowser():
    binary = TOOLS / "filebrowser"
    if not binary.is_file():
        url = "https://github.com/filebrowser/filebrowser/releases/download/v2.63.0/linux-amd64-filebrowser.tar.gz"
        with urlopen(url, timeout=120) as response:
            with tarfile.open(fileobj=response, mode="r|gz") as archive:
                for member in archive:
                    if member.name.lstrip("./") == "filebrowser" and member.isfile():
                        with archive.extractfile(member) as payload, binary.with_suffix(".part").open("wb") as output:
                            shutil.copyfileobj(payload, output)
                        binary.with_suffix(".part").replace(binary)
                        binary.chmod(0o755)
                        break
        if not binary.is_file():
            raise RuntimeError("FileBrowser binary missing from the release archive.")
    user = FILEBROWSER_USER
    password = FILEBROWSER_PASSWORD
    private_dir = Path(TemporaryDirectoryHandle.name)
    database = private_dir / "filebrowser.db"
    shared = WORK / "ilaria-shared"
    shared.mkdir(exist_ok=True)
    if shared.is_symlink():
        raise RuntimeError("The FileBrowser export folder must not be a symlink.")
    base = [str(binary), "--database", str(database)]
    run(*base, "config", "init", "--auth.method=json", stdout=subprocess.DEVNULL)
    run(*base, "users", "add", user, password, "--perm.admin", stdout=subprocess.DEVNULL)
    port = free_port()
    process, log = background([*base, "--root", str(shared), "--address", "127.0.0.1", "--port", str(port)], "filebrowser")
    deadline = time.monotonic() + 25
    while time.monotonic() < deadline and process.poll() is None:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                return port, user, password
        except OSError:
            time.sleep(0.5)
    raise RuntimeError(f"FileBrowser failed to start. See {log}.")

if TUNNEL not in {"Ngrok", "Gradio + Cloudflare", "Horizon"}:
    raise ValueError("Select Ngrok, Gradio + Cloudflare or Horizon.")
if TUNNEL == "Horizon" and not HORIZON_ID.strip():
    raise ValueError("Enter your Horizon ID before starting.")
if TUNNEL == "Ngrok" and not NGROK_AUTHTOKEN.strip():
    print("Ngrok Authtoken non inserito: passaggio automatico a Gradio + Cloudflare.", flush=True)
    TUNNEL = "Gradio + Cloudflare" 
TemporaryDirectoryHandle = TemporaryDirectory(prefix="ilaria-access-")
webui_password = WEBUI_PASSWORD
ui_process = globals().get("ui_process")
if ui_process is not None and ui_process.poll() is None:
    print("Restarting Ilaria RVC to load the current audio decoder.", flush=True)
    stop_ui(ui_process)

# Test the patched decoder in a fresh Python process before opening Gradio.
run(PYTHON, "-c", """
import tempfile
import wave
from pathlib import Path
from infer.lib.audio import load_audio

with tempfile.TemporaryDirectory() as directory:
    sample = Path(directory) / "audio check.wav"
    with wave.open(str(sample), "wb") as audio:
        audio.setparams((1, 2, 8000, 800, "NONE", "not compressed"))
        audio.writeframes(bytes(1600))
    decoded = load_audio(str(sample), 16000)
    assert decoded.shape == (1600,), decoded.shape
print("FFmpeg audio decoder verified.", flush=True)
""", cwd=ROOT, env=runtime_env)

if ui_process is None or ui_process.poll() is not None:
    launch_env = {**runtime_env, "PYTHONUNBUFFERED": "1", "ILARIA_WEBUI_PASSWORD": webui_password,
                  "ILARIA_KAGGLE_SHARE": "1" if TUNNEL == "Gradio + Cloudflare" else "0"}
    with log_file.open("w", encoding="utf-8") as log:
        ui_process = subprocess.Popen(
            [str(PYTHON), "-u", str(entrypoint), "--colab", "--noautoopen"],
            cwd=ROOT, env=launch_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
        )

print("Starting Ilaria RVC...", flush=True)
try:
    if TUNNEL == "Gradio + Cloudflare":
        public_url = wait_for_url(ui_process, log_file, r"https://[a-zA-Z0-9-]+\.(?:gradio\.live|gradio\.app)\b", 150)
    elif TUNNEL == "Horizon":
        if shutil.which("hrzn") is None:
            run("npm", "install", "-g", "@hrzn/cli", timeout=120)
        run("hrzn", "login", HORIZON_ID.strip(), timeout=180)
        local_url = wait_for_url(ui_process, log_file, r"http://127\.0\.0\.1:\d+", 120)
        process, log = background(["hrzn", "tunnel", local_url], "webui-horizon")
        public_url = wait_for_url(process, log, r"https://[a-zA-Z0-9-]+\.hrzn\.run(?:\s|$)").strip()
    elif TUNNEL == "Ngrok":
        run(sys.executable, "-m", "pip", "install", "--quiet", "pyngrok")
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
        local_url = wait_for_url(ui_process, log_file, r"http://127\.0\.0\.1:\d+", 120)
        local_port = int(re.search(r":(\d+)", local_url).group(1))
        try:
            ngrok.kill()
        except Exception:
            pass
        p_tunnel = ngrok.connect(local_port)
        public_url = p_tunnel.public_url

    port, filebrowser_user, filebrowser_password = start_filebrowser()
    filebrowser_local = f"http://127.0.0.1:{port}"
    if TUNNEL == "Gradio + Cloudflare":
        cloudflared = TOOLS / "cloudflared"
        download(
            "https://github.com/cloudflare/cloudflared/releases/download/2026.9.0/cloudflared-linux-amd64",
            cloudflared, "53b7a7a5420d188758d24341294acb0d1bca54296548ac05e38811a694ac6134",
        )
        cloudflared.chmod(0o755)
        command = [str(cloudflared), "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", filebrowser_local]
        pattern = r"https://[a-zA-Z0-9-]+\.trycloudflare\.com\b"
        process, log = background(command, "filebrowser-tunnel")
        filebrowser_url = wait_for_url(process, log, pattern).strip()
    elif TUNNEL == "Horizon":
        command = ["hrzn", "tunnel", filebrowser_local]
        pattern = r"https://[a-zA-Z0-9-]+\.hrzn\.run(?:\s|$)"
        process, log = background(command, "filebrowser-tunnel")
        filebrowser_url = wait_for_url(process, log, pattern).strip()
    elif TUNNEL == "Ngrok":
        try:
            fb_tunnel = ngrok.connect(port)
            filebrowser_url = fb_tunnel.public_url
        except Exception:
            cloudflared = TOOLS / "cloudflared"
            download(
                "https://github.com/cloudflare/cloudflared/releases/download/2026.9.0/cloudflared-linux-amd64",
                cloudflared, "53b7a7a5420d188758d24341294acb0d1bca54296548ac05e38811a694ac6134",
            )
            cloudflared.chmod(0o755)
            command = [str(cloudflared), "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", filebrowser_local]
            pattern = r"https://[a-zA-Z0-9-]+\.trycloudflare\.com\b"
            process, log = background(command, "filebrowser-tunnel")
            filebrowser_url = wait_for_url(process, log, pattern).strip()
except BaseException:
    stop_access()
    stop_ui(ui_process)
    TemporaryDirectoryHandle.cleanup()
    raise

display(HTML(f'<a href="{escape(filebrowser_url, quote=True)}" target="_blank" rel="noopener noreferrer">Open FileBrowser</a>'))
print(f"FileBrowser username: {filebrowser_user}")
print(f"FileBrowser password: {filebrowser_password}")
print("WebUI username: ilaria")
print(f"WebUI password: {webui_password}")
safe_url = escape(public_url, quote=True)
display(HTML(
    f'<a href="{safe_url}" target="_blank" rel="noopener noreferrer" '
    'style="display:inline-block;padding:10px 18px;border-radius:8px;'
    'background:#ffc0cb;color:#3f2340;font-weight:700;text-decoration:none">'
    'Open Ilaria RVC</a>'
    '<p>Keep this Kaggle session running while you use the app.</p>'
))
display(Javascript("window.open(" + repr(public_url) + ", '_blank', 'noopener,noreferrer');"))

# Export finished models independently of the browser tab.
from threading import Event, Lock, Thread

def start_model_export(source, destination, interval=5):
    stop = Event()
    lock = Lock()
    observed = {}

    def copy_models():
        with lock:
            for file in source.rglob("*"):
                if file.suffix.lower() not in {".pth", ".index"} or file.is_symlink():
                    continue
                if not file.is_file() or not file.resolve().is_relative_to(source.resolve()):
                    continue
                relative = file.relative_to(source)
                target = destination / relative
                if not target.resolve().is_relative_to(destination.resolve()):
                    continue
                before = file.stat()
                key = (before.st_size, before.st_mtime_ns)
                previous = observed.get(relative)
                observed[relative] = key
                if not key[0] or previous != key:
                    continue
                if target.is_file():
                    saved = target.stat()
                    if (saved.st_size, saved.st_mtime_ns) == key:
                        continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with tempfile.NamedTemporaryFile(dir=target.parent, prefix=".export-", delete=False) as handle:
                    temporary = Path(handle.name)
                try:
                    shutil.copy2(file, temporary)
                    after = file.stat()
                    if (after.st_size, after.st_mtime_ns) == key:
                        temporary.replace(target)
                finally:
                    temporary.unlink(missing_ok=True)

    def watch():
        while not stop.is_set():
            try:
                copy_models()
            except OSError as exc:
                print(f"Automatic model export will retry: {exc}", flush=True)
            stop.wait(interval)

    Thread(target=watch, name="ilaria-model-export", daemon=True).start()
    return stop, copy_models

if "model_export_stop" in globals():
    model_export_stop.set()
import tempfile
model_export_stop, export_models_now = start_model_export(
    MODELS, WORK / "ilaria-shared" / "models"
)

print("Streaming Ilaria RVC logs & session heartbeat...", flush=True)

try:
    with log_file.open("r", encoding="utf-8", errors="replace") as stream:
        stream.seek(0, os.SEEK_END)
        last_heartbeat = time.monotonic()
        while ui_process.poll() is None:
            line = stream.readline()
            if line:
                sys.stdout.write(line)
                sys.stdout.flush()
            else:
                time.sleep(1)
            if time.monotonic() - last_heartbeat >= 60:
                last_heartbeat = time.monotonic()
                now_str = time.strftime("%H:%M:%S")
                print(f"[{now_str}] Session active · Ilaria RVC running...", flush=True)
except KeyboardInterrupt:
    pass
finally:
    stop_access()
    stop_ui(ui_process)
    if "TemporaryDirectoryHandle" in globals():
        TemporaryDirectoryHandle.cleanup()
    if "model_export_stop" in globals():
        model_export_stop.set()
        export_models_now()
        export_models_now()


### Export results · optional

Set `EXPORT = True`. To include resumable checkpoints, click **Stop and save** in Ilaria RVC first and fill in `TRAINING_MODEL`.

In [ ]:
EXPORT = False
TRAINING_MODEL = ""  # Leave blank to exclude training files.

if EXPORT:
    from zipfile import ZipFile, ZIP_STORED
    from IPython.display import FileLink

    folders = [MODELS, ROOT / "InferOutput"]
    if TRAINING_MODEL:
        name = TRAINING_MODEL.strip()
        if not name or Path(name).name != name or name in {".", "..", "mute"} or "\\" in name:
            raise ValueError("Enter a model name instead of a path.")
        experiment = ROOT / "logs" / name
        if not experiment.is_dir():
            raise FileNotFoundError(f"Training files not found: {name}")
        folders.append(experiment)

    files = [
        file for folder in folders for file in folder.rglob("*")
        if file.is_file() and not file.is_symlink() and file.resolve().is_relative_to(folder.resolve())
        and (folder.name != "models" or file.suffix.lower() in {".pth", ".index"})
    ]
    if not files:
        print("There are no results to export.")
    else:
        export_folder = WORK / "ilaria-shared"
        export_folder.mkdir(exist_ok=True)
        archive = export_folder / "Ilaria-RVC-results.zip"
        partial = archive.with_suffix(".part")
        try:
            with ZipFile(partial, "w", compression=ZIP_STORED) as zip_file:
                for file in sorted(files):
                    zip_file.write(file, file.relative_to(REPO_ROOT) if file.is_relative_to(MODELS) else file.relative_to(ROOT))
            partial.replace(archive)
        finally:
            partial.unlink(missing_ok=True)
        link_path = os.path.relpath(archive, start=Path.cwd())
        display(FileLink(link_path))
        print(f"Also available in Output > {archive.name}")

### Stop or restart · optional

Save any running training first. Set `STOP = True` to stop the UI. Run step 4 again to restart.

In [ ]:
STOP = False
if STOP:
    if "stop_access" in globals():
        stop_access()
    stop_ui(globals().get("ui_process"))
    if "TemporaryDirectoryHandle" in globals():
        TemporaryDirectoryHandle.cleanup()
    if "model_export_stop" in globals():
        model_export_stop.set()
        export_models_now()
        export_models_now()
    print("Ilaria RVC stopped. Run section 4 to reopen it.")